In [4]:
import os
import cv2
import pandas as pd
import shutil

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/FETAL_PLANES"
target_path = r"/mnt/lustre/lalit47/down/class/FETAL_PLANES"

images_folder = os.path.join(source_path, "Images")
csv_path = os.path.join(source_path, "FETAL_PLANES_DB_data.csv")


def safe_class_name(class_name):
    """
    Clean class names so they are safe for folder names.
    Example: 'Fetal brain' -> 'Fetal_brain'
    """
    return str(class_name).strip().replace(" ", "_").replace("/", "_")


def find_image_file(image_name):
    """
    CSV image names do not include extension.
    This function finds the matching image file.
    """
    possible_extensions = [".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"]

    for ext in possible_extensions:
        image_path = os.path.join(images_folder, image_name + ext)
        if os.path.exists(image_path):
            return image_path

    return None


def process_and_save(src, dst):
    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)
    return True


def save_image(image_path, image_name, class_name, split_name):
    class_name = safe_class_name(class_name)

    class_out_dir = os.path.join(target_path, split_name, class_name)
    os.makedirs(class_out_dir, exist_ok=True)

    image_dst = os.path.join(class_out_dir, image_name + ".png")

    process_and_save(image_path, image_dst)


def create_dataset():
    df = pd.read_csv(csv_path, sep=";")

    # Remove extra spaces from column names
    df.columns = df.columns.str.strip()

    print("CSV columns found:")
    print(df.columns.tolist())

    required_columns = ["Image_name", "Plane", "Train"]
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Missing required column in CSV: {col}")

    train_count = 0
    test_count = 0
    val_count = 0
    missing_count = 0

    class_counts = {}

    for _, row in df.iterrows():
        image_name = str(row["Image_name"]).strip()
        class_name = str(row["Plane"]).strip()

        train_value = str(row["Train"]).strip()

        if train_value == "1":
            split_name = "train"
        else:
            split_name = "test"

        image_path = find_image_file(image_name)

        if image_path is None:
            print(f"Image not found for: {image_name}")
            missing_count += 1
            continue

        class_folder = safe_class_name(class_name)

        if class_folder not in class_counts:
            class_counts[class_folder] = {
                "train": 0,
                "test": 0,
                "val": 0
            }

        if split_name == "train":
            save_image(image_path, image_name, class_name, "train")

            train_count += 1
            class_counts[class_folder]["train"] += 1

        else:
            # Test images
            save_image(image_path, image_name, class_name, "test")

            # Same test images also copied to val
            save_image(image_path, image_name, class_name, "val")

            test_count += 1
            val_count += 1

            class_counts[class_folder]["test"] += 1
            class_counts[class_folder]["val"] += 1

    print("\nDataset created successfully!")
    print(f"Train images: {train_count}")
    print(f"Test images: {test_count}")
    print(f"Val images: {val_count}")
    print(f"Missing images: {missing_count}")

    print("\nClass-wise split:")
    for class_name, counts in class_counts.items():
        print(
            f"{class_name}: "
            f"train={counts['train']}, "
            f"test={counts['test']}, "
            f"val={counts['val']}"
        )

create_dataset()

CSV columns found:
['Image_name', 'Patient_num', 'Plane', 'Brain_plane', 'Operator', 'US_Machine', 'Train']

Dataset created successfully!
Train images: 7129
Test images: 5271
Val images: 5271
Missing images: 0

Class-wise split:
Other: train=2601, test=1612, val=1612
Maternal_cervix: train=981, test=645, val=645
Fetal_abdomen: train=353, test=358, val=358
Fetal_brain: train=1620, test=1472, val=1472
Fetal_femur: train=516, test=524, val=524
Fetal_thorax: train=1058, test=660, val=660
